# Time Series Forecasting: Market Data, Candlestick Charts, and Returns

**Target stock: MSFT** (chosen to match the group's Notebooks 3 and 4, and to satisfy the
assignment rule that NVDA may not be used as the target).

| Key | Value |
|:--|:--|
| **Course Codes** | BBT 4206 and BFS 4102 |
| **Course Names** | BBT 4206: Business Intelligence II (Week 7-9 of 13)<br>BFS 4102: Advanced Business Data Analytics (Week 1-3 of 13) |
| **Semester** | September to November 2026 |
| **Lecturer** | Allan Omondi |
| **Group** | Group B6 — this notebook is Member 1's contribution |
| **Note** | Educational use only. Nothing here is investment advice. |


## What you will learn in this notebook

1. Explain the Open, High, Low, Close and Volume (OHLCV) columns and read a candlestick chart.
2. Explain why the lab uses several predictors instead of only the stock's own past price.
3. Load a market-data snapshot for **MSFT** and check it for common data problems.
4. Explain the difference between **price**, **simple return**, and **log return**, and why the
   rest of the lab forecasts returns rather than price.

**Guiding question:** using only information known at today's close, how well can tomorrow's
return be forecast — and does any extra information beat simply assuming *tomorrow equals
today*? This comparison against the naive forecast is the thread that runs through every
notebook in this lab.


## Step 1: Set Up the Notebook

In [ ]:
import os
import sys
import subprocess

print("Python interpreter:", sys.executable)
print("Python version    :", sys.version.split()[0])

# Google Colab starts with an empty workspace, so we fetch the lab files first.
# This block does nothing in a local environment that already has the repo.
if "google.colab" in sys.modules and not os.path.exists("tslab"):
    subprocess.run(
        ["git", "clone", "https://github.com/course-files/TimeSeriesForecastingandNeuralNetworks.git"],
        check=True,
    )
    os.chdir("TimeSeriesForecastingandNeuralNetworks")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements/colab.txt"], check=True)
    print("Colab workspace prepared.")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mplfinance as mpf

from tslab import config
from tslab.data import load_market_data, data_quality_report

import warnings
warnings.filterwarnings("ignore")

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 5)


## Step 2: The Target and the Predictors

**Important:** before running this notebook, `tslab/config.py` must have
`TARGET_TICKER = "MSFT"` (not `NVDA`), so that the whole group — Notebooks 1, 2, 3 and 4 —
scores the same stock. Confirm this below before going any further.


In [ ]:
print("Target ticker    :", config.TARGET_TICKER)
print("Predictor tickers:", config.PREDICTOR_TICKERS)

assert config.TARGET_TICKER == "MSFT", (
    "config.TARGET_TICKER must be set to 'MSFT' in tslab/config.py "
    "to match the rest of the group's notebooks."
)


The seven predictors and the economic story behind each one:

| Predictor | Ticker | Why it might help forecast MSFT |
|:--|:--|:--|
| S&P 500 | ^GSPC | Broad US market — usually the single strongest predictor. |
| Tech sector ETF | XLK | MSFT's own sector; sector moves often lead individual names. |
| VIX ("fear index") | ^VIX | Expected volatility; tends to move opposite to stock prices. |
| 10-year Treasury yield | ^TNX | Rising yields can pressure technology valuations. |
| US dollar index | DX-Y.NYB | A strong dollar can pressure multinational revenue. |
| Brent crude | BZ=F | A broad macro/inflation signal. |
| Gold | GC=F | Often rises when confidence in stocks falls. |

Two rules matter here:

1. An economic story is only a **hypothesis** — the data, not the story, decides whether a
   predictor actually helps.
2. **Timing rule:** MSFT and the S&P 500 close at the same moment, so *today's* market move
   cannot forecast *today's* stock move. Only information known **up to and including today's
   close** may be used to forecast **tomorrow**.


## Step 3: Load the Data

In [ ]:
data = load_market_data()
print(f"Loaded {len(data)} trading days of {config.TARGET_TICKER}, "
      f"{data.index.min().date()} to {data.index.max().date()}, "
      f"with {len(config.PREDICTOR_TICKERS)} predictor series.")

data.head()


In [ ]:
data.tail()

## Step 4: Confirm the Loaded Data (Data Quality Checks)

Run these checks every time a new dataset is received, before modelling anything.


In [ ]:
print("Number of observations and variables:", data.shape)
data.info()


In [ ]:
data.describe()

In [ ]:
# tslab's own data-quality helper (missing values, duplicate dates, zero/negative
# prices, and other common problems in financial data).
data_quality_report(data)


## Step 5: Candlestick Chart

Each candle shows one trading day: the body spans **open-to-close** (colour shows whether the
day closed up or down), and the thin wicks above/below mark the day's **high** and **low**.
Volume is shown as a bar panel beneath the price.


In [ ]:
recent = data.tail(120)  # last ~6 months of trading days, for readability

mpf.plot(
    recent[["Open", "High", "Low", "Close", "Volume"]],
    type="candle",
    volume=True,
    style="yahoo",
    title=f"{config.TARGET_TICKER} — last {len(recent)} trading days",
    figratio=(16, 9),
)


## Step 6: Price, Simple Return, and Log Return

- **Simple return:** `(P_t - P_{t-1}) / P_{t-1}` — the plain percentage change.
- **Log return:** `ln(P_t / P_{t-1})` — approximately equal to the simple return for small moves,
  but **additive across time** (the sum of daily log returns equals the log return over the whole
  period) and more symmetric for gains vs losses. This is why the rest of the lab forecasts
  **log returns**, not price.


In [ ]:
price = data["Close"]
simple_return = price.pct_change()
log_return = np.log(price / price.shift(1))

returns = pd.DataFrame({"simple_return": simple_return, "log_return": log_return}).dropna()
returns.describe()


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

axes[0].plot(price.index, price, color="tab:blue")
axes[0].set_title(f"{config.TARGET_TICKER} closing price")

axes[1].plot(simple_return.index, simple_return, color="tab:orange", linewidth=0.7)
axes[1].set_title("Simple daily return")

axes[2].plot(log_return.index, log_return, color="tab:green", linewidth=0.7)
axes[2].set_title("Log daily return")

plt.tight_layout()
plt.show()


Notice the shape of each plot: the **price** panel drifts upward with no fixed level to
return to (a random walk — this is examined formally in Notebook 2), while the two **return**
panels hover around zero with a roughly constant spread throughout. That visual contrast is
exactly the stationarity argument Notebook 2 tests statistically with ADF and KPSS.


## Step 7: Summary and Reflection

- MSFT's price series looks like a random walk: no fixed mean, a level that drifts over time.
- Both return series oscillate around zero with a roughly stable spread — the visual signature
  of a stationary series.
- This is why the lab forecasts **returns**, not price: a forecasting model needs a target whose
  statistical behaviour does not change arbitrarily over time.
- **Next notebook (Notebook 2):** confirm this stationarity argument formally with ADF/KPSS
  tests, rolling statistics, ACF/PACF, a spurious-regression demonstration, and GARCH volatility
  clustering.
